<!-- Last changed: Facebook_20260824A -->
> **Enclave notebook — `Facebook_YYYYMMDD<A-Z>.ipynb` series.** This file is *authoring
> only*. It imports `metacontentlibraryapi`, which exists only inside Meta's Secure
> Research Environment; it cannot be executed, tested, or run against the API from
> this repo. Committed with **all outputs cleared** — Meta permits no data egress, so
> nothing produced by running it may be saved back here.
>
> Source of record: `Claude_Reference/H5N1_Aggregate_20260819.ipynb` (local, gitignored).

# H5N1 Farm-Page Post Aggregation — Meta Content Library

Successor to `H5N1_20260819.ipynb`. Same page/post discovery, but the post-level
term match becomes a **flag rather than a filter**, and the notebook ends in a single
aggregate table at the grain **(PageCategory, State, Zip, Date)**.

### Aggregate output columns
| Column | Meaning |
|---|---|
| `n_posts` | all posts in the cell |
| `n_posts_term` | posts whose text matches a `PostText` term |
| `n_engagement_ge10` | posts where `like_count + sad_count >= 10` |
| `n_ratio_defined` / `n_ratio_undefined` | posts with `like+sad > 0` / `== 0` |
| `n_sentiment_scored` | posts with text that scored |
| `like_count_total` / `sad_count_total` | summed reactions |
| `mean_like_ratio` | mean of `like/(like+sad)` over `n_ratio_defined` posts |
| `mean_sentiment` / `median_sentiment` | compound sentiment over `n_sentiment_scored` posts |

The decile bucket columns (`lr_*`, `sent_*`) have been removed. Distributional shape is
now carried only by the mean/median summaries — if a downstream model needs the shape
back, add quantiles to the `.agg()` rather than re-introducing fixed buckets.

### `RUN_MODE` — and why the two modes use *different API mechanisms*

There is no "full search" endpoint. Meta exposes two distinct mechanisms
([search guide](https://developers.facebook.com/docs/content-library-and-api/content-library-api/guides/search-guide/)):

| | `RUN_MODE = "preview"` | `RUN_MODE = "full"` |
|---|---|---|
| mechanism | **synchronous** search | **asynchronous** job |
| call | `client.get(path="facebook/posts/preview")` | `client.post(path="facebook/posts/job")` |
| paging | `query_next_page()`, 10 results/page | job runs server-side; `get_data()` when `COMPLETE` |
| hard limit | **1,000 results per search** | **100,000 results per job** |
| latency | seconds | **minutes to days** |
| rate limit | 60 queries/min | **1 query/min** |

Both draw on the same **500,000 records / 7-day rolling** query budget, so a full run is
not free — use `facebook/posts/estimate` (the `RUN_ESTIMATE` toggle) before committing.

`RUN_MODE = "full"` is therefore **resumable, not blocking**: it submits one job per
surface batch, polls for a bounded wall-clock budget, and if the jobs are still running
it prints the job IDs and stops. Re-run the same cell later — `POST_JOBS` survives in
the kernel and nothing is resubmitted.

> ⚠️ **Page discovery has no async variant.** `facebook/pages/preview` is synchronous,
> so any *single* search is capped at 1,000 results in both modes. Discovery is
> therefore partitioned — one search per category, sub-split by description term for
> any category that still hits the cap — and every slice is recorded in `page_slices`.
> Slices still flagged `capped` there are truncated by an unknown amount: the page
> universe is a measured sample, not a census. See § 1.

### Four things to know before you trust the numbers
1. **`n_posts` is a true "every post these pages made" denominator in the default
   mode.** `POST_QUERY_MODE = "all_posts"` sends no `q`. If the API rejects a `q`-less
   query, switch to `"broad_terms"` and read `n_posts` as "broad-term posts," not
   "all posts."
2. **Zip and State are the *page's* registered location, not the author's.** Meta
   exposes no author geography. See the `meta-content-library-constraints` memory.
3. **Sentiment is lexicon-based and computed in-enclave.** No model weights are
   downloaded — the enclave has no egress. See the sentiment cell for the fallback chain.
4. **Nothing here leaves the enclave without Meta's output review.** The final cell
   applies small-cell suppression to make the aggregate reviewable; raw post text is
   never part of the export shape.
### 0 — Checkpointing, and why it exists

An async pull outlives the session that started it. `JOB_MODE = "SNAPSHOT"` means Meta
holds the results for roughly a year, so the thing that must survive a session timeout
is the **job ID**, not the rows: lose `POST_JOBS` and you resubmit, re-spending the
500,000-record / 7-day query budget. Lose the harvested rows and you re-harvest for
free.

§ 0 therefore writes run state to the enclave's own filesystem under `CKPT_DIR`, and
`_ckpt_init` reads it back, so re-running a cell keeps what it already has instead of
re-fetching it. Page discovery — hours of synchronous, rate-limited calls — is skipped
outright when its rows are already on disk.

> ⚠️ **A checkpoint is not an export.** These files never leave the enclave, and
> writing one is not a substitute for Meta's output review. Same boundary as
> `job.write_data_to_file()`: a within-enclave convenience, nothing more.


In [ ]:
# Last changed: Facebook_20260824A
from metacontentlibraryapi import MetaContentLibraryAPIClient as client
from datetime import date, timedelta
import math
import os
import re
import time
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

# ----------------------------------------------------------------------------
# PAGE DISCOVERY (who is posting)
# ----------------------------------------------------------------------------
# Full category list is now the default: PageCategory is a grouping dimension in
# the final table, so collapsing to ['Farm'] would make that column constant.
PageCategories = [
    "Farm", "Agricultural Service", "Agricultural Cooperative", "Agriculture",
    "Veterinarian", "Animal Shelter", "Wildlife Sanctuary", "Zoo",
]
# PageCategories = ["Farm"]   # <- the old simplified run

PageDescription = [
    "poultry", "chicken", "turkey", "egg", "hatchery",
    "flock", "livestock", "farm", "farming", "agriculture",
    "agricultural", "ranch", "dairy", "veterinary", "veterinarian",
    "avian",
]
PageCountry_Code = "US"

# ----------------------------------------------------------------------------
# POST DISCOVERY (what they wrote)
# ----------------------------------------------------------------------------
# PostText = the *precise* concept list. Multi-word entries are phrases, which the
# API cannot search, so they are enforced client-side as the has_post_term flag.
PostText = [
    "avian influenza", "avian flu", "chicken flu", "bird flu", "poultry flu",
    "zoonotic flu", "duck flu", "farm flu",
    "h5n1", "h5n8", "h7n9", "h9n2", "h10n3",
    "hpai",   # Highly Pathogenic Avian Influenza
    "lpai",   # Low Pathogenic Avian Influenza
]

# PostSingleWordText = the *recall* list sent to the API (single tokens only).
# Only reaches the API when POST_QUERY_MODE == "broad_terms".
PostSingleWordText = [
    "influenza", "flu",
    "h5n1", "h5n8", "h7n9", "h9n2", "h10n3",
    "hpai", "lpai",
]

PostStartDate = "2022-01-01"
PostEndDate   = "2026-07-31"

# ----------------------------------------------------------------------------
# RUN MODE  —  preview smoke test vs. the full pull
# ----------------------------------------------------------------------------
# These are two DIFFERENT API mechanisms, not one endpoint with two names:
#
#   "preview" -> SYNCHRONOUS search. client.get(path="facebook/posts/preview").
#                10 results/page via query_next_page(), hard-capped at 1,000
#                results per search. Seconds. Also truncates page discovery and
#                the date window so the whole notebook runs in minutes.
#
#   "full"    -> ASYNCHRONOUS job. client.post(path="facebook/posts/job").
#                Up to 100,000 results per job, but takes MINUTES TO DAYS and is
#                rate-limited to 1 submission/minute. Submits one job per surface
#                batch, then polls. Resumable: re-run the cell to keep polling.
#
# Docs: /docs/content-library-and-api/content-library-api/guides/search-guide/
RUN_MODE = "preview"          # "preview" | "full"

# Documented endpoint paths. There is no "facebook/posts/search".
PAGES_PREVIEW_PATH   = "facebook/pages/preview"    # sync only - no async variant
POSTS_PREVIEW_PATH   = "facebook/posts/preview"    # sync,  <= 1,000 results
POSTS_JOB_PATH       = "facebook/posts/job"        # async, <= 100,000 results
POSTS_ESTIMATE_PATH  = "facebook/posts/estimate"   # rough result-size estimate

# --- page discovery ---------------------------------------------------------
# Sync search is capped at 1,000 results, and facebook/pages has no async variant,
# so discovery is PARTITIONED: one search per category, and any category that
# still hits the cap is re-run split across PageDescription terms. Page rows draw
# on the same 500,000-record / 7-day budget as the post jobs.
PARTITION_PAGE_SEARCH = True   # False = one unpartitioned search, capped at 1,000
PAGE_SEARCH_SLEEP_S   = 1.05   # sync limit is 60 queries/min; each paged call counts

# --- preview-mode throttles -------------------------------------------------
PREVIEW_MAX_SURFACES  = 250    # pages fed to the post pull (= one SURFACE_BATCH)
PREVIEW_MAX_API_PAGES = 25     # 25 pages x 10 results = up to 250 pages discovered
PREVIEW_WINDOW_DAYS   = 60     # trailing days of the configured post window

# --- full-mode (async job) settings ----------------------------------------
# SNAPSHOT: results retrievable for up to a year, periodically re-redacted.
# LIVE:     available for 30 days at most. SNAPSHOT is the reproducible choice
#           for a capstone that has to be re-run months later.
JOB_MODE            = "SNAPSHOT"
JOB_NAME            = "H5N1 farm-page posts"
JOB_DESCRIPTION     = ("H5N1 early-warning capstone: posts from US agriculture / "
                       "veterinary / animal-facility Pages.")
JOB_SUBMIT_SLEEP_S  = 65       # async rate limit is 1 query/min - stay under it
JOB_POLL_INTERVAL_S = 60       # seconds between status checks
JOB_POLL_BUDGET_MIN = 30       # stop polling after this long and print job IDs
JOB_RESULT_CAP      = 100_000  # documented async ceiling; used for the cap check

# Estimate result size before spending query budget (full mode only).
# Budget is 500,000 records per 7-day rolling window, shared with sync searches.
RUN_ESTIMATE = True

# The estimate is a GATE, not a printout. While RUN_ESTIMATE is on and this stays
# False, the post cell prints the estimate and STOPS without submitting anything.
# Read the estimate, then set this True and re-run that cell to actually submit.
CONFIRM_SUBMIT_JOBS = False

# ----------------------------------------------------------------------------
# OTHER TOGGLES
# ----------------------------------------------------------------------------
# "all_posts"   -> no q at all: n_posts is a TRUE denominator. This is the default.
# "broad_terms" -> q = PostSingleWordText (recall-filtered denominator, faster).
#                  Fall back to this only if the API rejects a q-less query.
POST_QUERY_MODE = "all_posts"

# Restrict post collection to primary-category == Farm pages only.
RESTRICT_TO_FARM_ONLY = False

# Reliability threshold for like_count + sad_count (requirement 4).
ENGAGEMENT_MIN = 10

# Small-cell suppression applied to the export-shaped table (0 disables).
MIN_CELL_POSTS = 0

# ----------------------------------------------------------------------------
# CHECKPOINTING  -  surviving a session timeout
# ----------------------------------------------------------------------------
# Written to the enclave's OWN disk. This is not an export: nothing crosses the
# Meta boundary, and no checkpoint substitutes for output review.
CKPT_DIR = os.path.join(os.getcwd(), "h5n1_ckpt")

# True  -> a re-run keeps in-memory state, else reloads it from CKPT_DIR.
# False -> ignore checkpoints and start clean, which RE-SPENDS query budget.
RESTORE_FROM_CHECKPOINT = True

# Poll rounds of JOB_POLL_BUDGET_MIN each, checkpointing after every one. The
# default is ~12 h of unattended polling; a session timeout then costs at most
# one round, not the run.
POLL_ROUNDS = 24


def build_ORterms(terms):
    """
    Search terms need to be concatenated with | as an OR term.

    The API matches token-by-token with no phrase support, so a multi-word term
    must become an AND group. That creates a term-distance problem which is fixed
    client-side by the has_post_term regex below.

    Reserved chars (& | - ( ) and space) are not allowed inside a keyword.
    """
    groups = []
    for t in terms:
        tokens = [tok for tok in re.split(r"[^0-9A-Za-z]+", t) if tok]
        if tokens:
            groups.append(tokens[0] if len(tokens) == 1 else "(" + " & ".join(tokens) + ")")
    return " | ".join(groups)


# `until` must be strictly in the past or the API returns 400 / subcode 3790079.
_yesterday = (date.today() - timedelta(days=1)).isoformat()
if PostEndDate >= date.today().isoformat():
    print(f"PostEndDate {PostEndDate} is not in the past -> clamped to {_yesterday}")
    PostEndDate = _yesterday

# ----------------------------------------------------------------------------
# Resolve the run mode. Everything downstream reads the resolved names below,
# never the raw RUN_MODE config.
# ----------------------------------------------------------------------------
if RUN_MODE not in ("preview", "full"):
    raise ValueError(f"RUN_MODE must be 'preview' or 'full', got {RUN_MODE!r}")
if POST_QUERY_MODE not in ("all_posts", "broad_terms"):
    raise ValueError("POST_QUERY_MODE must be 'all_posts' or 'broad_terms', "
                     f"got {POST_QUERY_MODE!r}")
if JOB_MODE not in ("SNAPSHOT", "LIVE"):
    raise ValueError(f"JOB_MODE must be 'SNAPSHOT' or 'LIVE', got {JOB_MODE!r}")

IS_PREVIEW = RUN_MODE == "preview"

MAX_SURFACES  = PREVIEW_MAX_SURFACES  if IS_PREVIEW else None
MAX_API_PAGES = PREVIEW_MAX_API_PAGES if IS_PREVIEW else None

RUN_END = PostEndDate
if IS_PREVIEW:
    _pv_start = date.fromisoformat(PostEndDate) - timedelta(days=PREVIEW_WINDOW_DAYS - 1)
    RUN_START = max(PostStartDate, _pv_start.isoformat())
else:
    RUN_START = PostStartDate

print(f"run mode   : {RUN_MODE}"
      + ("   <-- SMOKE TEST, counts are not the real dataset" if IS_PREVIEW
         else f"   <-- async jobs, {JOB_MODE} mode, minutes to days"))
print(f"pages      : {PAGES_PREVIEW_PATH} (sync, 1,000-result ceiling in BOTH modes)")
print(f"posts      : {POSTS_PREVIEW_PATH if IS_PREVIEW else POSTS_JOB_PATH}"
      f"  ({'sync, 1,000/search' if IS_PREVIEW else 'async, 100,000/job'})")
print(f"categories : {len(PageCategories)}")
print(f"post window: {RUN_START} .. {RUN_END}"
      + (f"   (configured: {PostStartDate} .. {PostEndDate})" if IS_PREVIEW else ""))
print(f"surfaces   : {'all' if MAX_SURFACES is None else f'first {MAX_SURFACES}'}")
print(f"query mode : {POST_QUERY_MODE}")

<!-- Last changed: Facebook_20260824A -->
## 0 — Checkpoint machinery

Everything below runs against the enclave's local filesystem. Three pieces:

* `_ckpt_write` / `_ckpt_read` — one object to one file. Raw API rows go to gzipped
  JSONL so no `json_normalize` assumption is baked into the file; DataFrames go to
  gzipped pickle; small dicts and lists go to plain JSON.
* `_ckpt_init(NAME, default)` — resolve module state in priority order: what is already
  in memory, else what is on disk, else a fresh default. This is what makes a re-run
  cheap and a resumed session possible.
* `checkpoint(tag)` — persist every piece of run state that currently exists. Safe to
  call at any point, including before the expensive cells have ever run.

`_ckpt_write` never raises. A checkpoint that kills a twelve-hour run is worse than no
checkpoint at all.

**Resuming a lost session:** re-run the config cell and § 0, leave
`RESTORE_FROM_CHECKPOINT = True`, then run down the notebook. Page discovery skips,
`submit_post_jobs` skips every batch already in `POST_JOBS`, and polling reattaches to
the jobs Meta is still holding.


In [ ]:
# Last changed: Facebook_20260824A
# ---------------------------------------------------------------------------
# Checkpoints are written to the enclave's own filesystem and nowhere else.
# ---------------------------------------------------------------------------
import gzip
import json

os.makedirs(CKPT_DIR, exist_ok=True)

# Names persisted as gzipped JSONL: raw API payloads, kept in their original
# shape so a reload is not hostage to how json_normalize behaved that day.
_CKPT_ROWSETS = {"page_rows", "post_rows"}

# Persist order: cheapest and most irreplaceable first. POST_JOBS is a few hundred
# bytes and is the only handle on the SNAPSHOT results Meta holds for a year; the
# DataFrames at the end are all recomputable from POST_ROWS.
_CKPT_STATE = ("POST_JOBS", "PAGE_SLICES", "POST_BATCH", "PAGE_ROWS", "POST_ROWS",
               "SourcePages", "CleanUSPages", "FarmPosts",
               "AggByCatStateZipDate", "ExportAgg", "ExportAggState")


def _ckpt_write(name, obj):
    """Best-effort persist. NEVER raises - a checkpoint must not kill a long run."""
    try:
        if name in _CKPT_ROWSETS:
            p = os.path.join(CKPT_DIR, name + ".jsonl.gz")
            with gzip.open(p, "wt", encoding="utf-8") as fh:
                for r in obj:
                    fh.write(json.dumps(r, default=str) + "\n")
        elif isinstance(obj, pd.DataFrame):
            p = os.path.join(CKPT_DIR, name + ".pkl.gz")
            obj.to_pickle(p, compression="gzip")
        else:
            p = os.path.join(CKPT_DIR, name + ".json")
            with open(p, "w", encoding="utf-8") as fh:
                json.dump(obj, fh, default=str, indent=2)
        return p
    except Exception as e:
        print(f"  !! checkpoint FAILED for {name}: {type(e).__name__}: {e}")
        return None


def _ckpt_read(name):
    """Load one checkpointed object, or None if it is absent or unreadable."""
    for ext in (".jsonl.gz", ".pkl.gz", ".json"):
        p = os.path.join(CKPT_DIR, name + ext)
        if not os.path.exists(p):
            continue
        try:
            if ext == ".jsonl.gz":
                with gzip.open(p, "rt", encoding="utf-8") as fh:
                    return [json.loads(ln) for ln in fh if ln.strip()]
            if ext == ".pkl.gz":
                return pd.read_pickle(p, compression="gzip")
            with open(p, encoding="utf-8") as fh:
                return json.load(fh)
        except Exception as e:
            print(f"  !! checkpoint UNREADABLE {name}{ext}: {type(e).__name__}: {e}")
    return None


def _ckpt_init(varname, default):
    """In-memory state, else the checkpoint, else a fresh default.

    Re-running a cell must not re-fetch what has already been paid for, and a
    fresh kernel must be able to pick up where the dead one stopped.
    """
    if not RESTORE_FROM_CHECKPOINT:
        return default
    live = globals().get(varname)
    if live is not None and len(live):
        print(f"  {varname:<12} kept in memory      ({len(live):,})")
        return live
    disk = _ckpt_read(varname.lower())
    if disk is not None and len(disk):
        print(f"  {varname:<12} restored from disk  ({len(disk):,})")
        return disk
    return default


def _ckpt_manifest(tag):
    """Provenance. A checkpoint you cannot attribute to a run config is noise."""
    return {
        "tag": tag,
        "written_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "run_mode": globals().get("RUN_MODE"),
        "job_mode": globals().get("JOB_MODE"),
        "run_start": str(globals().get("RUN_START")),
        "run_end": str(globals().get("RUN_END")),
        "query_mode": globals().get("POST_QUERY_MODE"),
        "sentiment_engine": globals().get("SENTIMENT_ENGINE"),
        "farm_only": globals().get("RESTRICT_TO_FARM_ONLY"),
        "engagement_min": globals().get("ENGAGEMENT_MIN"),
        "n_surfaces": len(globals().get("SourcePages", [])),
        "submit_gated": globals().get("SUBMIT_GATED"),
        "all_jobs_done": globals().get("ALL_JOBS_DONE"),
    }


def checkpoint(tag=""):
    """Persist whatever run state exists right now. Safe to call at any point."""
    _ckpt_write("run_manifest", _ckpt_manifest(tag))
    total = 0
    for name in _CKPT_STATE:
        obj = globals().get(name)
        if obj is None or not len(obj):
            continue
        p = _ckpt_write(name.lower(), obj)
        if p:
            total += os.path.getsize(p)
            print(f"  {name:<22} {len(obj):>9,}  ->  {os.path.basename(p)}")
    print(f"  checkpoint '{tag}' -> {CKPT_DIR}  ({total / 1e6:.1f} MB)")


print(f"checkpoint dir : {CKPT_DIR}")
print(f"restore        : {RESTORE_FROM_CHECKPOINT}")
_existing = sorted(os.listdir(CKPT_DIR))
print(f"on disk        : {_existing if _existing else 'nothing yet'}")


## Sentiment engine

The enclave has no outbound network, so a `transformers` pipeline that pulls weights
from the Hub will fail. The chain below degrades gracefully:

1. `vaderSentiment` (pip package, lexicon ships inside the wheel) — preferred.
2. `nltk` VADER — only if `vader_lexicon` is already present on disk.
3. **Inline lexicon** — defined in this cell, always works, zero dependencies.

All three return a compound score in `[-1, +1]` using VADER's normalization, so
`mean_sentiment` / `median_sentiment` are comparable whichever engine wins. Print
`SENTIMENT_ENGINE` after running and record it in the methods write-up — a run scored by fallback #3 is not
interchangeable with one scored by #1.

Emoji are deliberately **not** scored: the prior cohort found emoji added noise rather
than signal.

In [ ]:
# ---------------------------------------------------------------------------
# Inline fallback lexicon. Small, domain-tilted, VADER-style valences on [-4, +4].
# ---------------------------------------------------------------------------
_LEX = {
    # --- outbreak / loss (domain-specific negatives) ---
    "cull": -2.6, "culled": -2.8, "culling": -2.8, "depopulate": -3.0,
    "depopulated": -3.0, "depopulation": -3.0, "euthanize": -2.8,
    "euthanized": -2.8, "euthanasia": -2.8, "outbreak": -2.6, "outbreaks": -2.6,
    "infected": -2.4, "infection": -2.3, "infectious": -1.8, "contagious": -1.8,
    "quarantine": -2.0, "quarantined": -2.0, "lockdown": -1.8, "biosecurity": -0.4,
    "dead": -2.9, "death": -2.9, "deaths": -2.9, "died": -2.8, "dying": -2.8,
    "mortality": -2.5, "carcass": -2.2, "carcasses": -2.2, "sick": -2.2,
    "illness": -2.0, "disease": -2.0, "diseased": -2.4, "virus": -1.6,
    "viral": -1.0, "epidemic": -2.8, "pandemic": -2.8, "spread": -1.0,
    "spreading": -1.2, "positive": -0.5, "confirmed": -0.6, "detected": -0.8,
    "detection": -0.8, "loss": -2.4, "losses": -2.4,
    "lost": -2.2, "devastating": -3.4, "devastated": -3.3, "tragedy": -3.2,
    "heartbreaking": -3.3, "heartbroken": -3.3, "crisis": -2.8, "emergency": -2.4,
    "shortage": -2.0, "recall": -1.6, "banned": -1.8, "ban": -1.6,
    "restriction": -1.4, "restrictions": -1.4, "closed": -1.4, "shut": -1.4,
    "struggle": -2.2, "struggling": -2.2, "suffering": -2.8, "risk": -1.4,
    "danger": -2.4, "dangerous": -2.6, "threat": -2.4, "fear": -2.4,
    "afraid": -2.2, "scared": -2.4, "worried": -2.2, "worry": -2.0,
    "concern": -1.4, "concerned": -1.6, "panic": -2.8, "anxious": -2.0,
    # --- general negatives ---
    "bad": -2.5, "awful": -3.0, "terrible": -3.1, "horrible": -3.2,
    "worst": -3.4, "sad": -2.6, "angry": -2.6, "hate": -3.0, "hard": -1.2,
    "difficult": -1.6, "problem": -1.8, "problems": -1.8, "fail": -2.4,
    "failed": -2.4, "failure": -2.6, "damage": -2.2, "destroy": -3.0,
    "destroyed": -3.0, "sorry": -1.2, "unfortunately": -1.8, "never": -1.0,
    # --- general positives ---
    "good": 1.9, "great": 3.1, "excellent": 3.2, "amazing": 3.2,
    "wonderful": 3.1, "fantastic": 3.2, "best": 3.2, "love": 3.2,
    "loved": 2.9, "happy": 2.7, "happiness": 2.8, "joy": 2.8, "excited": 2.4,
    "exciting": 2.4, "beautiful": 2.8, "proud": 2.4, "grateful": 2.6,
    "thank": 2.2, "thanks": 2.2, "thankful": 2.5, "blessed": 2.6,
    "welcome": 1.8, "enjoy": 2.2, "enjoyed": 2.2, "fun": 2.3, "nice": 1.8,
    "safe": 1.6, "safety": 1.0, "healthy": 2.2, "health": 0.8, "recovered": 2.2,
    "recovery": 1.8, "improving": 1.8, "improved": 2.0, "success": 2.7,
    "successful": 2.7, "support": 1.8, "supporting": 1.8, "help": 1.6,
    "helping": 1.6, "hope": 1.9, "hopeful": 2.2, "fresh": 1.6, "delicious": 2.6,
    "open": 1.0, "available": 0.8, "new": 0.6, "clean": 1.4, "strong": 1.8,
    "protect": 1.2, "protected": 1.6, "prevent": 0.8, "vaccine": 0.6,
    "vaccinated": 1.0,
}

_NEGATIONS = {
    "not", "no", "never", "none", "nobody", "nothing", "neither", "nowhere",
    "cannot", "cant", "wont", "dont", "doesnt", "didnt", "isnt", "arent",
    "wasnt", "werent", "hasnt", "havent", "hadnt", "aint", "without",
}
_BOOSTERS = {
    "very": 1.45, "really": 1.4, "extremely": 1.6, "so": 1.3, "absolutely": 1.5,
    "completely": 1.5, "totally": 1.45, "incredibly": 1.55, "super": 1.4,
    "highly": 1.4, "massively": 1.5, "entire": 1.3, "whole": 1.25,
    "slightly": 0.7, "somewhat": 0.75, "kinda": 0.7, "barely": 0.6,
    "hardly": 0.6, "little": 0.75, "sort": 0.8, "marginally": 0.7,
}
_TOKEN_RE = re.compile(r"[a-z0-9']+")


def _inline_compound(text):
    """VADER-shaped compound score on [-1, 1] from the inline lexicon."""
    if not text:
        return np.nan
    toks = _TOKEN_RE.findall(str(text).lower())
    if not toks:
        return np.nan
    total, found = 0.0, 0
    for i, tok in enumerate(toks):
        val = _LEX.get(tok)
        if val is None:
            continue
        found += 1
        # intensifier immediately before
        if i > 0 and toks[i - 1] in _BOOSTERS:
            val *= _BOOSTERS[toks[i - 1]]
        # negation within the preceding 3 tokens
        if any(t in _NEGATIONS for t in toks[max(0, i - 3):i]):
            val *= -0.74
        total += val
    if found == 0:
        return 0.0
    return float(np.clip(total / math.sqrt(total * total + 15.0), -1.0, 1.0))


# ---------------------------------------------------------------------------
# Engine selection
# ---------------------------------------------------------------------------
SENTIMENT_ENGINE = None
_scorer = None

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as _SIA
    _an = _SIA()
    _an.polarity_scores("a good test")            # prove it runs
    _scorer = lambda t: _an.polarity_scores(str(t))["compound"] if t else np.nan
    SENTIMENT_ENGINE = "vaderSentiment"
except Exception as _e1:
    try:
        from nltk.sentiment.vader import SentimentIntensityAnalyzer as _SIA
        _an = _SIA()
        _an.polarity_scores("a good test")        # raises if lexicon absent
        _scorer = lambda t: _an.polarity_scores(str(t))["compound"] if t else np.nan
        SENTIMENT_ENGINE = "nltk-vader"
    except Exception as _e2:
        _scorer = _inline_compound
        SENTIMENT_ENGINE = "inline-lexicon"


def score_sentiment(series):
    """Compound sentiment on [-1, 1]; NaN for empty/absent text."""
    s = series.fillna("").astype(str)
    out = s.map(lambda t: _scorer(t) if t.strip() else np.nan)
    return pd.to_numeric(out, errors="coerce")


print(f"SENTIMENT_ENGINE = {SENTIMENT_ENGINE}")
print(score_sentiment(pd.Series([
    "We lost the entire flock to bird flu. Absolutely devastating.",
    "Fresh eggs available today, the girls are healthy and happy!",
    "Routine biosecurity update for the barn.",
    "",
])).round(3).tolist())

## 1 — Pages

Only Pages carry geography, so page discovery has to come first. Country and category
are filtered client-side, not in the request: the API's `admin_countries` is about who
administers the Page, and `categories` is not an exact-primary match.

**Discovery is partitioned, because it has to be.** `facebook/pages/preview` is
synchronous-only — there is no `facebook/pages/job` — so any single search is capped at
1,000 results, in *both* run modes. One search across all categories therefore returns a
truncated sample, silently. The cell below instead runs:

1. one search per `PageCategories` entry (8 searches), then
2. for any category that *still* returns 1,000 rows, a re-run split across the 16
   `PageDescription` terms,

deduplicating by page id and recording every slice in `page_slices`.

⚠️ **This widens the sample; it does not guarantee a census.** A slice like
`Farm` × `farm` can exceed 1,000 on its own, and the API exposes no state-level page
filter to subdivide further. Any slice still flagged `capped` in `page_slices` is
truncated by an unknown amount — that is what the sanity check reports, and it belongs
in the methods write-up. Set `PARTITION_PAGE_SEARCH = False` to go back to one
capped search.

In [ ]:
# Last changed: Facebook_20260824A
# Page discovery is SYNCHRONOUS ONLY - there is no facebook/pages/job endpoint -
# so every individual search inherits the 1,000-result ceiling, in both run modes.
SYNC_SEARCH_CAP = 1000

PAGE_FIELDS = ("id,name,page_categories,about,follower_count,"
               "location.country_code,location.state,location.city,location.zip")


def search_pages(categories, q, max_api_pages=None):
    """One synchronous page search, paginated. Returns (rows, hit_cap).

    Paging stops at SYNC_SEARCH_CAP because the API will not serve past it; a
    slice that reaches the cap is TRUNCATED, not complete, and says so.
    """
    params = {"q": q,
              "admin_countries": "US",       # comma list, not operators
              "fields": PAGE_FIELDS,
              "categories": categories}

    resp = client.get(path=PAGES_PREVIEW_PATH, params=params)
    rows = list(resp.json().get("data", []))
    n_api = 1

    while client.has_next_page(resp) and len(rows) < SYNC_SEARCH_CAP:
        if max_api_pages is not None and n_api >= max_api_pages:
            break
        time.sleep(PAGE_SEARCH_SLEEP_S)      # stay under 60 queries/min
        resp = client.query_next_page(resp)
        rows.extend(resp.json().get("data", []))
        n_api += 1

    return rows, len(rows) >= SYNC_SEARCH_CAP


# Discovery is the slowest cell in the notebook - synchronous, rate-limited, and
# hours long at full scope. Restoring it rather than repeating it is the whole
# point of the checkpoint layer.
PAGE_ROWS   = _ckpt_init("PAGE_ROWS", [])    # every row from every slice, pre-dedup
PAGE_SLICES = _ckpt_init("PAGE_SLICES", [])  # one record per search, so truncation is measured not guessed

_q_pages = build_ORterms(PageDescription)


def _run_slice(cat, term, q):
    """Run one slice, record it, and return whether it hit the cap."""
    try:
        rows, capped = search_pages(cat, q, MAX_API_PAGES)
    except Exception as e:
        PAGE_SLICES.append({"category": cat, "term": term, "rows": 0,
                            "capped": None, "sub_split": False,
                            "error": f"{type(e).__name__}: {e}"})
        print(f"  {cat:<24} {term:<14} FAILED  {type(e).__name__}: {e}")
        return False

    PAGE_ROWS.extend(rows)
    PAGE_SLICES.append({"category": cat, "term": term, "rows": len(rows),
                        "capped": capped, "sub_split": False})
    print(f"  {cat:<24} {term:<14} {len(rows):>5,} rows"
          f"{'  <-- CAPPED' if capped else ''}")
    return capped


if PAGE_ROWS:
    print(f"{len(PAGE_ROWS):,} page rows already present - skipping discovery.\n"
          f"Set RESTORE_FROM_CHECKPOINT = False to force a fresh search.\n")
elif PARTITION_PAGE_SEARCH:
    print(f"partitioned page discovery: {len(PageCategories)} category slices"
          + ("  (preview: no sub-splitting)" if IS_PREVIEW else "")
          + "\n")
    for _cat in PageCategories:
        _capped_cat = _run_slice(_cat, "<all terms>", _q_pages)
        _parent = len(PAGE_SLICES) - 1

        # Only a category that is still truncated is worth re-querying per term.
        if _capped_cat and not IS_PREVIEW:
            print(f"    -> {_cat} is capped; sub-splitting across "
                  f"{len(PageDescription)} terms")
            # The parent slice is EXPECTED to be capped - that is what triggered
            # the sub-split - so it is excluded from the truncation report below.
            PAGE_SLICES[_parent]["sub_split"] = True
            for _term in PageDescription:
                _run_slice(_cat, _term, build_ORterms([_term]))
else:
    print("unpartitioned page discovery - capped at 1,000 rows\n")
    _run_slice(",".join(PageCategories), "<all terms>", _q_pages)

page_slices = pd.DataFrame(PAGE_SLICES)

NormalizedRows = (pd.json_normalize(PAGE_ROWS).drop_duplicates("id")
                  if PAGE_ROWS else pd.DataFrame(columns=["id"]))

print(f"\n{len(PAGE_ROWS):,} rows across {len(page_slices)} slice(s) "
      f"-> {len(NormalizedRows):,} unique pages")

# A sub-split parent is superseded by its term slices; only unresolved truncation counts.
_capped_slices = page_slices[(page_slices["capped"] == True)
                             & (~page_slices["sub_split"])]
if len(_capped_slices):
    print(f"\n!! {len(_capped_slices)} slice(s) still at the {SYNC_SEARCH_CAP:,} cap - "
          f"truncated by an unknown amount:")
    print(_capped_slices[["category", "term", "rows"]].to_string(index=False))
    print("!! The page universe is a SAMPLE in these slices, not a census. "
          "Record this in the methods write-up.")
else:
    print("\nno unresolved slice hit the cap")

_failed_slices = page_slices[page_slices["capped"].isna()]
if len(_failed_slices):
    print(f"\n!! {len(_failed_slices)} slice(s) errored and contributed no pages - "
          f"that is silent truncation too:")
    print(_failed_slices[["category", "term", "error"]].to_string(index=False))
elif not len(_capped_slices):
    print("the page universe is complete for these categories and terms")

checkpoint("pages-discovered")


In [ ]:
# Country and PRIMARY-category filtering happen HERE, not in the request.
us = NormalizedRows[NormalizedRows["location.country_code"].fillna("") == "US"]

want = {c.lower() for c in PageCategories}


def primary_category(cell):
    """First element of page_categories, or None."""
    items = cell if isinstance(cell, (list, tuple)) else []
    return str(items[0]) if items else None


us = us.assign(page_category=us["page_categories"].apply(primary_category))
CleanUSPages = us[us["page_category"].fillna("").str.lower().isin(want)].copy()

FarmPages = CleanUSPages[CleanUSPages["page_category"].str.lower() == "farm"]

# Which page set drives the post pull.
SourcePages = FarmPages if RESTRICT_TO_FARM_ONLY else CleanUSPages

# Preview mode truncates the surface list so the post pull stays a single batch.
if MAX_SURFACES is not None and len(SourcePages) > MAX_SURFACES:
    print(f"{RUN_MODE}: capping surfaces {len(SourcePages):,} -> {MAX_SURFACES}")
    SourcePages = SourcePages.head(MAX_SURFACES)

print(f"{len(NormalizedRows):,} pages -> {len(us):,} US-located "
      f"-> {len(CleanUSPages):,} with a wanted PRIMARY category "
      f"-> {len(FarmPages):,} Farms")
print(f"sourcing posts from {len(SourcePages):,} pages "
      f"({'Farm only' if RESTRICT_TO_FARM_ONLY else 'all wanted categories'})\n")
print(CleanUSPages["page_category"].value_counts())

## 2 — Posts

`FarmPosts` is kept **whole**. The precise `PostText` concepts are applied later as a
flag, never as a row filter — that is the point of this notebook.

The next cell only *defines* the fetch machinery; the cell after it runs it. Which
path executes depends on `RUN_MODE`:

* **preview** → `client.get("facebook/posts/preview")` per surface batch, paginated,
  capped at 1,000 results per batch.
* **full** → `client.post("facebook/posts/job")` per surface batch, then polling.
  Submissions are spaced by `JOB_SUBMIT_SLEEP_S` because async search is limited to
  **one query per minute**.

Both keep state in module-level `POST_ROWS` / `POST_BATCH` / `POST_JOBS`, so a
re-run resumes rather than restarting. Surface batches stay at 250 — that is the
documented maximum number of IDs per request.

In [ ]:
# Last changed: Facebook_20260824A
SURFACE_BATCH = 250          # documented max IDs per request
CAP = SYNC_SEARCH_CAP        # sync ceiling; async jobs use JOB_RESULT_CAP

POST_FIELDS = "id,creation_time,text,lang,surface.id,post_owner_id,post_owner_type,statistics"

# Module-level so state survives a crash mid-run; _ckpt_init extends that to
# surviving the kernel itself. JSON has no integer keys, so POST_JOBS batch
# indices come back as strings and are coerced here.
POST_ROWS  = _ckpt_init("POST_ROWS", [])    # raw post payloads
POST_BATCH = _ckpt_init("POST_BATCH", [])   # per-batch diagnostics
POST_JOBS  = {int(k): v for k, v in
              _ckpt_init("POST_JOBS", {}).items()}  # batch -> job metadata


def _epoch(d, end_of_day=False):
    """The async job endpoint's documented samples pass Unix seconds, not ISO dates."""
    ts = pd.Timestamp(d, tz="UTC")
    if end_of_day:
        ts = ts + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return int(ts.timestamp())


def _base_params(chunk, q, fields):
    p = {"lang": "en",
         "search_scope": "post_text_only",
         "surface_ids": chunk,
         "fields": fields}
    if q:                        # omitted entirely in "all_posts" mode
        p["q"] = q
    return p


def _batches(page_ids):
    page_ids = list(page_ids)
    for i in range(0, len(page_ids), SURFACE_BATCH):
        yield i // SURFACE_BATCH, page_ids[i:i + SURFACE_BATCH]


# ---------------------------------------------------------------------------
# SYNCHRONOUS path (RUN_MODE == "preview")
# ---------------------------------------------------------------------------
def fetch_posts_sync(page_ids, q, since, until, fields=POST_FIELDS):
    """Paginated sync search per surface batch. Resumable - rerun skips done batches."""
    done = {b["batch"] for b in POST_BATCH if b.get("returned") is not None}

    for b, chunk in _batches(page_ids):
        if b in done:
            print(f"  batch {b:>2}  skip (already done)")
            continue
        try:
            params = _base_params(chunk, q, fields)
            params["since"], params["until"] = since, until

            resp = client.get(path=POSTS_PREVIEW_PATH, params=params)
            recs = list(resp.json().get("data", []))
            while client.has_next_page(resp):
                resp = client.query_next_page(resp)
                recs.extend(resp.json().get("data", []))

            POST_ROWS.extend(recs)
            POST_BATCH.append({"batch": b, "surfaces": len(chunk),
                               "returned": len(recs), "capped": len(recs) >= CAP})
            print(f"  batch {b:>2}  {len(recs):>5,} posts"
                  f"{'  <-- CAPPED' if len(recs) >= CAP else ''}"
                  f"   running {len(POST_ROWS):,}")

        except Exception as e:
            POST_BATCH.append({"batch": b, "surfaces": len(chunk),
                               "returned": None, "capped": None,
                               "error": f"{type(e).__name__}: {e}"})
            print(f"  batch {b:>2}  FAILED  {type(e).__name__}: {e}")


# ---------------------------------------------------------------------------
# ASYNCHRONOUS path (RUN_MODE == "full")
# ---------------------------------------------------------------------------
def _job_id(payload):
    """Pull the job id out of a /job POST response.

    The docs display the response payload but never name the key, so try the
    plausible shapes and fail loudly with the raw payload rather than guessing.
    """
    if hasattr(payload, "json"):
        payload = payload.json()
    if isinstance(payload, dict):
        for k in ("id", "job_id", "async_job_id", "request_id"):
            if payload.get(k):
                return str(payload[k])
        for k in ("data", "job", "result"):
            inner = payload.get(k)
            if isinstance(inner, list) and inner:
                inner = inner[0]
            if isinstance(inner, dict):
                for kk in ("id", "job_id", "async_job_id"):
                    if inner.get(kk):
                        return str(inner[kk])
    raise KeyError(f"no job id found in submit response: {payload!r}")


def _status_str(st):
    """Normalize get_status() output - dict, response, or bare string."""
    if hasattr(st, "json"):
        try:
            st = st.json()
        except Exception:
            pass
    if isinstance(st, dict):
        for k in ("status", "state", "job_status"):
            if st.get(k):
                return str(st[k]).upper()
    return str(st).upper()


def _job_records(data):
    """Normalize get_data() output to a list of post dicts."""
    if hasattr(data, "json"):
        try:
            data = data.json()
        except Exception:
            pass
    if isinstance(data, dict):
        for k in ("data", "records", "results"):
            if isinstance(data.get(k), list):
                return data[k]
        return []
    return list(data) if isinstance(data, (list, tuple)) else []


def _estimate_count(est):
    """Pull a numeric count out of an /estimate payload of unknown shape."""
    if hasattr(est, "json"):
        try:
            est = est.json()
        except Exception:
            return None
    candidates = [est]
    if isinstance(est, dict):
        for k in ("data", "result", "estimate"):
            v = est.get(k)
            if isinstance(v, list) and v:
                v = v[0]
            if isinstance(v, dict):
                candidates.append(v)
    for c in candidates:
        if isinstance(c, dict):
            for k in ("estimated_count", "estimated_results", "count", "estimate",
                      "total_count", "upper_bound"):
                v = c.get(k)
                if isinstance(v, bool):
                    continue
                if isinstance(v, (int, float)):
                    return int(v)
    return None


def estimate_posts(page_ids, q, since, until, fields=POST_FIELDS):
    """Rough result-size estimate before spending the 500k/7-day query budget.

    Returns (total, n_unparsed). A non-zero n_unparsed means the payload shape is
    not one _estimate_count recognizes, so the total is a floor - read the raw
    payloads printed per batch instead of trusting it.
    """
    total, unparsed = 0, 0
    for b, chunk in _batches(page_ids):
        try:
            params = _base_params(chunk, q, fields)
            params["since"], params["until"] = since, until
            est = client.get(path=POSTS_ESTIMATE_PATH, params=params).json()
            n = _estimate_count(est)
            print(f"  batch {b:>2}  estimate: {est}"
                  + ("" if n is not None else "   <-- count not parsed"))
            if n is None:
                unparsed += 1
            else:
                total += n
        except Exception as e:
            unparsed += 1
            print(f"  batch {b:>2}  estimate FAILED  {type(e).__name__}: {e}")
    return total, unparsed


def submit_post_jobs(page_ids, q, since, until, fields=POST_FIELDS):
    """One async job per surface batch. Already-submitted batches are skipped."""
    for b, chunk in _batches(page_ids):
        if b in POST_JOBS:
            print(f"  batch {b:>2}  already submitted as job {POST_JOBS[b]['job_id']}")
            continue
        try:
            params = _base_params(chunk, q, fields)
            params["since"] = _epoch(since)
            params["until"] = _epoch(until, end_of_day=True)
            params["mode"] = JOB_MODE
            params["name"] = f"{JOB_NAME} [batch {b}]"
            params["description"] = JOB_DESCRIPTION

            resp = client.post(path=POSTS_JOB_PATH, params=params)
            if not POST_JOBS:                 # show the real payload shape once
                print(f"  submit response payload: {resp.json()}")
            POST_JOBS[b] = {"job_id": _job_id(resp), "surfaces": len(chunk),
                            "status": "IN_PROGRESS", "n": None}
            print(f"  batch {b:>2}  submitted job {POST_JOBS[b]['job_id']}")
        except Exception as e:
            POST_BATCH.append({"batch": b, "surfaces": len(chunk),
                               "returned": None, "capped": None,
                               "error": f"submit: {type(e).__name__}: {e}"})
            print(f"  batch {b:>2}  SUBMIT FAILED  {type(e).__name__}: {e}")

        time.sleep(JOB_SUBMIT_SLEEP_S)        # async limit is 1 query/minute


def collect_post_jobs(budget_min=None):
    """Poll submitted jobs, harvesting COMPLETE ones. Returns True when all resolved.

    Bounded by a wall-clock budget because jobs can take days - when the budget
    runs out this returns False and the caller reports the outstanding job IDs.
    Re-running is safe: harvested batches are recorded in POST_BATCH and skipped.
    """
    budget_min = JOB_POLL_BUDGET_MIN if budget_min is None else budget_min
    deadline = time.time() + budget_min * 60
    harvested = {b["batch"] for b in POST_BATCH if b.get("returned") is not None}

    while True:
        pending = [b for b in POST_JOBS if b not in harvested]
        if not pending:
            return True

        for b in list(pending):
            meta = POST_JOBS[b]
            try:
                job = client.get_async_job(job_id=meta["job_id"])
                status = _status_str(job.get_status())
                meta["status"] = status

                if status.startswith("COMPLETE"):
                    recs = _job_records(job.get_data())
                    POST_ROWS.extend(recs)
                    meta["n"] = len(recs)
                    POST_BATCH.append({"batch": b, "surfaces": meta["surfaces"],
                                       "returned": len(recs),
                                       "capped": len(recs) >= JOB_RESULT_CAP,
                                       "job_id": meta["job_id"]})
                    harvested.add(b)
                    print(f"  batch {b:>2}  COMPLETE  {len(recs):>6,} posts"
                          f"{'  <-- CAPPED' if len(recs) >= JOB_RESULT_CAP else ''}"
                          f"   running {len(POST_ROWS):,}")

                elif status.startswith("FAIL") or status.startswith("ERROR"):
                    POST_BATCH.append({"batch": b, "surfaces": meta["surfaces"],
                                       "returned": None, "capped": None,
                                       "job_id": meta["job_id"],
                                       "error": f"job status {status}"})
                    harvested.add(b)
                    print(f"  batch {b:>2}  FAILED  job {meta['job_id']} -> {status}")

            except Exception as e:
                print(f"  batch {b:>2}  poll error  {type(e).__name__}: {e}")

        pending = [b for b in POST_JOBS if b not in harvested]
        if not pending:
            return True
        if time.time() >= deadline:
            print(f"\n  polling budget ({budget_min} min) exhausted; "
                  f"{len(pending)} job(s) still running:")
            for b in pending:
                print(f"    batch {b:>2}  {POST_JOBS[b]['job_id']}  "
                      f"{POST_JOBS[b]['status']}")
            print("  re-run this cell later - POST_JOBS is kept, nothing resubmits.")
            return False

        print(f"  {len(pending)} job(s) still running; "
              f"sleeping {JOB_POLL_INTERVAL_S}s")
        time.sleep(JOB_POLL_INTERVAL_S)

<!-- Last changed: Facebook_20260824A -->
### Pre-flight — checkpoint and job status

Read this before running the pull. If `POST_JOBS` came back from a checkpoint the jobs
below are already submitted and already paid for — the next cell skips submission and
goes straight to polling. A `COMPLETE` job whose rows are not yet in `POST_ROWS` costs
nothing but the time to fetch it.


In [ ]:
# Last changed: Facebook_20260824A
print(f"checkpoint dir : {CKPT_DIR}")
for _f in sorted(os.listdir(CKPT_DIR)):
    print(f"  {_f:<28} {os.path.getsize(os.path.join(CKPT_DIR, _f)) / 1e6:>8.2f} MB")

_harvested = {b["batch"] for b in POST_BATCH if b.get("returned") is not None}
print(f"\nPAGE_ROWS {len(PAGE_ROWS):,}   POST_ROWS {len(POST_ROWS):,}   "
      f"POST_JOBS {len(POST_JOBS)}   harvested batches {len(_harvested)}")

if POST_JOBS and not IS_PREVIEW:
    print("\nlive job status (one status call each, no result budget spent):")
    for _b, _m in sorted(POST_JOBS.items()):
        try:
            _st = _status_str(client.get_async_job(job_id=_m["job_id"]).get_status())
        except Exception as e:
            _st = f"UNREACHABLE ({type(e).__name__})"
        _m["status"] = _st
        print(f"  batch {_b:>2}  {_m['job_id']}  {_st}"
              f"{'   <-- rows already harvested' if _b in _harvested else ''}")
else:
    print("\nno submitted jobs on record")


In [ ]:
# Last changed: Facebook_20260824A
_q = None if POST_QUERY_MODE == "all_posts" else build_ORterms(PostSingleWordText)
if _q is None:
    print("POST_QUERY_MODE='all_posts' (default): no q sent, so n_posts is a true\n"
          "denominator. If every batch FAILS with a 400, the endpoint requires a\n"
          "query - set POST_QUERY_MODE='broad_terms' and rerun.\n")
else:
    print("POST_QUERY_MODE='broad_terms': n_posts counts broad-term posts only.\n")

ALL_JOBS_DONE = True     # meaningful in full mode only
SUBMIT_GATED  = False    # True when the estimate gate stopped the run

if IS_PREVIEW:
    print(f"SYNC pull from {POSTS_PREVIEW_PATH}  ({RUN_START} .. {RUN_END})")
    fetch_posts_sync(SourcePages["id"], q=_q, since=RUN_START, until=RUN_END)

else:
    n_batches = math.ceil(len(SourcePages) / SURFACE_BATCH)

    # --- estimate first, and treat it as a gate ---------------------------
    if RUN_ESTIMATE and not POST_JOBS:
        print(f"ESTIMATE via {POSTS_ESTIMATE_PATH} "
              f"({n_batches} batch(es)) - spends no result budget\n")
        _est, _unparsed = estimate_posts(SourcePages["id"], q=_q,
                                         since=RUN_START, until=RUN_END)
        print(f"\n  estimated total: {_est:,} posts across {n_batches} batch(es)"
              f"   (budget: 500,000 records / rolling 7 days)")
        if _unparsed:
            print(f"  !! {_unparsed} batch estimate(s) unparsed - the total above is "
                  f"a FLOOR. Read the payloads printed per batch.")
        print(f"  !! Scope: {len(SourcePages):,} surfaces drawn from a page universe "
              f"capped at {SYNC_SEARCH_CAP:,}.")
        print(f"  !! This is NOT an estimate of all US farm-page posts - see the page "
              f"discovery cell.")

        if not CONFIRM_SUBMIT_JOBS:
            SUBMIT_GATED = True
            ALL_JOBS_DONE = False
            print("\nSTOPPED before submitting - no result budget has been spent.\n"
                  "Set CONFIRM_SUBMIT_JOBS = True in the config cell, re-run it, "
                  "then re-run this cell to submit.")

    # --- submit + poll ----------------------------------------------------
    if not SUBMIT_GATED:
        print(f"\nASYNC submit to {POSTS_JOB_PATH}  ({RUN_START} .. {RUN_END})")
        print(f"  {n_batches} batch(es) x {JOB_SUBMIT_SLEEP_S}s spacing "
              f"~ {n_batches * JOB_SUBMIT_SLEEP_S / 60:.0f} min just to submit\n")
        submit_post_jobs(SourcePages["id"], q=_q, since=RUN_START, until=RUN_END)

        # Job ids before anything else: they are the only handle on results Meta
        # already holds, and losing them means re-spending the query budget.
        checkpoint("post-submit")

        print(f"\npolling: up to {POLL_ROUNDS} round(s) x "
              f"{JOB_POLL_BUDGET_MIN} min "
              f"(~{POLL_ROUNDS * JOB_POLL_BUDGET_MIN / 60:.1f} h), "
              f"checkpointing after each")
        for _round in range(POLL_ROUNDS):
            print(f"\n  -- poll round {_round + 1}/{POLL_ROUNDS} --")
            ALL_JOBS_DONE = collect_post_jobs()
            checkpoint(f"poll-{_round + 1}")
            if ALL_JOBS_DONE:
                break

batch_report = pd.DataFrame(POST_BATCH)
FarmPosts = pd.json_normalize(POST_ROWS).drop_duplicates("id")

_capped = int(batch_report.get("capped", pd.Series(dtype=float)).fillna(False).sum())
_failed = int(batch_report.get("returned", pd.Series(dtype=float)).isna().sum())
_cap_used = CAP if IS_PREVIEW else JOB_RESULT_CAP

print(f"\n{len(FarmPosts):,} unique posts from {len(SourcePages):,} pages")
print(f"{_capped} of {len(batch_report)} batches hit the {_cap_used:,} cap; "
      f"{_failed} batches failed")
if SUBMIT_GATED:
    print("\n!! Estimate gate is closed - no posts fetched. Cells below will run on "
          "an EMPTY frame.")
elif not ALL_JOBS_DONE:
    print("\n!! Jobs are still running - FarmPosts is PARTIAL. "
          "Re-run this cell before trusting anything below.")

checkpoint("posts-fetched")


### 2a — Term flag (requirement 1)

`build_ORterms` sent single tokens to the API because there is no phrase search, so a
post containing "flu" anywhere qualifies. The regex below re-imposes the real
`PostText` concepts — multi-word terms as adjacency patterns — and records the result
as `has_post_term` + `matched_terms`. **No rows are dropped.**

In [ ]:
def term_re(term):
    """Multi-word terms become adjacency patterns; single tokens stay bounded tokens."""
    return re.compile(
        r"\b" + r"[\s\-/]+".join(map(re.escape, term.split())) + r"\b",
        re.IGNORECASE,
    )


TERMS = {t: term_re(t) for t in PostText}

_txt = FarmPosts["text"].fillna("") if "text" in FarmPosts.columns else pd.Series("", index=FarmPosts.index)
hits = pd.DataFrame({t: _txt.str.contains(rx) for t, rx in TERMS.items()},
                    index=FarmPosts.index)

FarmPosts["has_post_term"] = hits.any(axis=1)
FarmPosts["matched_terms"] = hits.apply(lambda r: ", ".join(hits.columns[r]), axis=1)

# Kept only for continuity with the old notebook - not used downstream.
FarmPostsFiltered = FarmPosts.loc[FarmPosts["has_post_term"]]

print(f"{len(FarmPosts):,} posts kept; "
      f"{int(FarmPosts['has_post_term'].sum()):,} "
      f"({FarmPosts['has_post_term'].mean():.1%}) match a PostText term")
print(hits.sum().sort_values(ascending=False).to_string())

## 3 — Attach page geography and category (requirement 2)

`page_category` / `state` / `zip` describe the **posting Page**, not the author. Meta
exposes no author geography and no county, which is why the modeling grain for social
signal is state-or-coarser. Zip is included because it is the finest thing available,
not because it is a reliable locator of the person who wrote the post.

In [ ]:
PageLookup = (
    SourcePages[["id", "name", "page_category",
                 "location.state", "location.city", "location.zip"]]
    .rename(columns={"id": "page_id",
                     "name": "page_name",
                     "location.state": "state",
                     "location.city": "city",
                     "location.zip": "zip"})
)

FarmPosts = FarmPosts.merge(PageLookup, left_on="surface.id", right_on="page_id", how="left")

# Grouping keys must not be NaN or the cells silently vanish from the aggregate.
FarmPosts["page_category"] = FarmPosts["page_category"].fillna("UNKNOWN")
FarmPosts["state"] = FarmPosts["state"].fillna("").replace("", "UNKNOWN")
FarmPosts["zip"] = (FarmPosts["zip"].fillna("").astype(str).str.strip()
                    .str.split("-").str[0].replace("", "UNKNOWN"))

FarmPosts["post_date"] = pd.to_datetime(
    FarmPosts["creation_time"], errors="coerce", utc=True
).dt.tz_convert(None).dt.date

_unmatched = FarmPosts["page_id"].isna().sum()
print(f"{_unmatched:,} posts did not join to a page "
      f"({_unmatched / max(len(FarmPosts), 1):.1%})")
print(f"{FarmPosts['post_date'].isna().sum():,} posts have an unparseable creation_time")
print(f"{(FarmPosts['zip'] == 'UNKNOWN').mean():.1%} of posts have no page zip")

## 4 — Engagement, reliability flag, and sentiment (requirements 3-6)

Post-level measures only. The decile bucketing that used to live here is gone — the
aggregate carries `mean_like_ratio`, `mean_sentiment`, and `median_sentiment` instead
of twenty count columns.

In [ ]:
def _num(df, col):
    """Numeric column or a zero column if the API omitted it."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series(0.0, index=df.index)


FarmPosts["like_count"] = _num(FarmPosts, "statistics.like_count")
FarmPosts["sad_count"]  = _num(FarmPosts, "statistics.sad_count")
FarmPosts["like_sad"]   = FarmPosts["like_count"] + FarmPosts["sad_count"]

# --- requirement 4: reliability flag ---------------------------------------
FarmPosts["engagement_ge10"] = FarmPosts["like_sad"] >= ENGAGEMENT_MIN

# --- requirement 5: like share of like+sad ----------------------------------
# NaN where like+sad == 0: an undefined ratio, not a zero one.
FarmPosts["like_ratio"] = np.where(
    FarmPosts["like_sad"] > 0,
    FarmPosts["like_count"] / FarmPosts["like_sad"].replace(0, np.nan),
    np.nan,
)

# --- requirement 6: sentiment ----------------------------------------------
FarmPosts["sentiment"] = score_sentiment(
    FarmPosts["text"] if "text" in FarmPosts.columns else pd.Series("", index=FarmPosts.index)
)

print(f"like+sad >= {ENGAGEMENT_MIN}: {int(FarmPosts['engagement_ge10'].sum()):,} "
      f"({FarmPosts['engagement_ge10'].mean():.1%}) of {len(FarmPosts):,} posts")
print(f"like_ratio defined for {int(FarmPosts['like_ratio'].notna().sum()):,} posts "
      f"(mean {FarmPosts['like_ratio'].mean():.3f})")
print(f"sentiment scored for {int(FarmPosts['sentiment'].notna().sum()):,} posts "
      f"via {SENTIMENT_ENGINE} (mean {FarmPosts['sentiment'].mean():.3f})")

## 5 — The aggregate table

One row per **(PageCategory, State, Zip, Date)**. Every `n_*` column is a count of
posts, so `n_posts` is the denominator for `n_posts_term` and `n_engagement_ge10`, and
`n_ratio_defined + n_ratio_undefined == n_posts`.

`mean_like_ratio` averages only the posts where the ratio is defined; `mean_sentiment`
and `median_sentiment` average only the posts that scored. Both are NaN in a cell where
no post qualified — that is correct, and must not be filled with 0.

In [ ]:
GRP = ["page_category", "state", "zip", "post_date"]

AggByCatStateZipDate = (
    FarmPosts
    .groupby(GRP, dropna=False)
    .agg(
        n_posts             = ("id", "size"),
        n_posts_term        = ("has_post_term", "sum"),
        n_engagement_ge10   = ("engagement_ge10", "sum"),
        n_ratio_defined     = ("like_ratio", "count"),
        n_sentiment_scored  = ("sentiment", "count"),
        like_count_total    = ("like_count", "sum"),
        sad_count_total     = ("sad_count", "sum"),
        mean_like_ratio     = ("like_ratio", "mean"),
        mean_sentiment      = ("sentiment", "mean"),
        median_sentiment    = ("sentiment", "median"),
    )
)

AggByCatStateZipDate["n_ratio_undefined"] = (
    AggByCatStateZipDate["n_posts"] - AggByCatStateZipDate["n_ratio_defined"]
)

_int_cols = ["n_posts", "n_posts_term", "n_engagement_ge10", "n_ratio_defined",
             "n_ratio_undefined", "n_sentiment_scored",
             "like_count_total", "sad_count_total"]
_float_cols = ["mean_like_ratio", "mean_sentiment", "median_sentiment"]

AggByCatStateZipDate[_int_cols] = AggByCatStateZipDate[_int_cols].astype(int)

AggByCatStateZipDate = (
    AggByCatStateZipDate[_int_cols + _float_cols]
    .reset_index()
    .sort_values(GRP)
    .reset_index(drop=True)
)

print(f"{len(AggByCatStateZipDate):,} aggregate rows "
      f"covering {AggByCatStateZipDate['n_posts'].sum():,} posts")
AggByCatStateZipDate.head(20)

## 6 — Sanity checks

Run these before believing anything. Each one has failed at least once in a
comparable pipeline.

In [ ]:
checks = []

def chk(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})

a = AggByCatStateZipDate

chk("aggregate post count == FarmPosts row count",
    a["n_posts"].sum() == len(FarmPosts),
    f"{a['n_posts'].sum():,} vs {len(FarmPosts):,}")

chk("n_posts_term <= n_posts", (a["n_posts_term"] <= a["n_posts"]).all())
chk("n_engagement_ge10 <= n_posts", (a["n_engagement_ge10"] <= a["n_posts"]).all())
chk("n_sentiment_scored <= n_posts", (a["n_sentiment_scored"] <= a["n_posts"]).all())

chk("ratio defined + undefined == n_posts",
    (a["n_ratio_defined"] + a["n_ratio_undefined"] == a["n_posts"]).all())

chk("mean_like_ratio in [0, 1] where defined",
    a["mean_like_ratio"].dropna().between(0.0, 1.0).all())
chk("mean_like_ratio NaN exactly where n_ratio_defined == 0",
    (a["mean_like_ratio"].isna() == (a["n_ratio_defined"] == 0)).all())

chk("mean_sentiment in [-1, 1] where defined",
    a["mean_sentiment"].dropna().between(-1.0, 1.0).all())
chk("mean_sentiment NaN exactly where n_sentiment_scored == 0",
    (a["mean_sentiment"].isna() == (a["n_sentiment_scored"] == 0)).all())

chk("no NaN grouping keys",
    not a[GRP].isna().any().any())

# --- collection completeness -----------------------------------------------
chk("no batch failed", _failed == 0, f"{_failed} failed batches")
chk("no batch hit the result cap", _capped == 0,
    f"{_capped} capped batches at {_cap_used:,} - counts are floors, not totals")
chk("all async jobs finished", ALL_JOBS_DONE,
    "jobs still running - FarmPosts is partial" if not ALL_JOBS_DONE else "")

chk("post_date inside the requested window",
    (pd.Series(pd.to_datetime(a["post_date"], errors="coerce")).between(
        pd.Timestamp(RUN_START), pd.Timestamp(RUN_END)) | a["post_date"].isna()).all(),
    f"{RUN_START} .. {RUN_END} ({RUN_MODE})")

# Page discovery is sync-only in both modes, so slices can be silently truncated.
_n_capped_slices = int(((page_slices["capped"] == True)
                        & (~page_slices["sub_split"])).sum())
_n_failed_slices = int(page_slices["capped"].isna().sum())
chk("no page-discovery slice hit the sync cap", _n_capped_slices == 0,
    f"{_n_capped_slices} of {len(page_slices)} slices capped at "
    f"{SYNC_SEARCH_CAP:,} - the page universe is a sample there, not a census")
chk("no page-discovery slice failed", _n_failed_slices == 0,
    f"{_n_failed_slices} slice(s) errored")

chk("RUN_MODE is 'full'", RUN_MODE == "full",
    "preview output is a smoke test, not a result" if IS_PREVIEW else "")

pd.DataFrame(checks)

In [ ]:
# Where the mass actually sits - useful for deciding whether zip is a usable grain.
print("posts per aggregate cell:")
print(AggByCatStateZipDate["n_posts"].describe().round(2).to_string())
print(f"\ncells with a single post: "
      f"{(AggByCatStateZipDate['n_posts'] == 1).mean():.1%}")
print("\ntop categories by post volume:")
print(AggByCatStateZipDate.groupby("page_category")["n_posts"]
      .sum().sort_values(ascending=False).to_string())
print("\ntop states by post volume:")
print(AggByCatStateZipDate.groupby("state")["n_posts"]
      .sum().sort_values(ascending=False).head(15).to_string())

## 7 — Export shape

⚠️ **Nothing leaves the enclave without Meta's output review.** This cell only *builds*
the reviewable object; it deliberately does not write a file. It also refuses to treat a
preview run or an unfinished async pull as an exportable result — only a completed
`RUN_MODE = "full"` job set is one.

`MIN_CELL_POSTS` suppresses thin cells. Zip-level daily counts are exactly the kind of
grain a reviewer pushes back on, so set it to a real threshold (5 or 10) before
requesting export, and be ready to fall back to a state-level rollup.

The state rollup re-derives its means from the summed like/sad totals rather than
averaging the zip-level means, which would weight a one-post zip the same as a
five-hundred-post one. `mean_sentiment` cannot be recovered that way — it is dropped
from the rollup instead of being computed wrong.

If output review clears the aggregate, note that `job.write_data_to_file(directory=...,
filename=...)` writes **raw post rows**, not this table — it is a within-enclave
convenience, not an export route.

In [ ]:
if IS_PREVIEW:
    print("RUN_MODE='preview' - this is a smoke test, not an exportable result.\n"
          "Set RUN_MODE='full', rerun the notebook, then build the export.\n")
elif not ALL_JOBS_DONE:
    print("Async jobs are still running - this aggregate is PARTIAL.\n"
          "Re-run the post cell until every job is COMPLETE before exporting.\n")

ExportAgg = AggByCatStateZipDate.copy()

if MIN_CELL_POSTS > 0:
    _keep = ExportAgg["n_posts"] >= MIN_CELL_POSTS
    print(f"suppressing {int((~_keep).sum()):,} of {len(ExportAgg):,} cells "
          f"below n_posts={MIN_CELL_POSTS} "
          f"({int(ExportAgg.loc[~_keep, 'n_posts'].sum()):,} posts)")
    ExportAgg = ExportAgg[_keep].reset_index(drop=True)
else:
    print("MIN_CELL_POSTS=0 - no suppression applied. Set it before requesting export.")

# State-level fallback if zip-day is judged too granular by output review.
# Sum the counts, then re-derive the like ratio from the totals; a mean of means
# would weight a one-post zip the same as a five-hundred-post one.
ExportAggState = (
    AggByCatStateZipDate
    .drop(columns=["zip", "mean_like_ratio", "mean_sentiment", "median_sentiment"])
    .groupby(["page_category", "state", "post_date"], dropna=False)
    .sum(numeric_only=True)
    .reset_index()
)

_ls = ExportAggState["like_count_total"] + ExportAggState["sad_count_total"]
ExportAggState["like_ratio_total"] = np.where(
    _ls > 0, ExportAggState["like_count_total"] / _ls.replace(0, np.nan), np.nan
)

print(f"\nExportAgg      : {len(ExportAgg):,} rows x {ExportAgg.shape[1]} cols")
print(f"ExportAggState : {len(ExportAggState):,} rows x {ExportAggState.shape[1]} cols "
      f"(no sentiment - it cannot be re-derived from totals)")

print(f"\nrun_mode={RUN_MODE}  window={RUN_START}..{RUN_END}  "
      f"engine={SENTIMENT_ENGINE}  query_mode={POST_QUERY_MODE}  "
      f"farm_only={RESTRICT_TO_FARM_ONLY}  engagement_min={ENGAGEMENT_MIN}")
if not IS_PREVIEW:
    print(f"job_mode={JOB_MODE}  jobs={len(POST_JOBS)}  all_complete={ALL_JOBS_DONE}")
    for _b, _m in sorted(POST_JOBS.items()):
        print(f"  batch {_b:>2}  {_m['job_id']}  {_m['status']}  n={_m['n']}")

ExportAgg.head(20)

<!-- Last changed: Facebook_20260824A -->
## 8 — Final checkpoint

Persist the derived frames too. They are all recomputable from `POST_ROWS`, so this is
convenience rather than insurance — but it is the difference between reopening the
notebook tomorrow and re-running §§ 2–6 first.

Same boundary as everywhere else: this writes inside the enclave. `ExportAgg` becoming a
file here does **not** make it exported.


In [ ]:
# Last changed: Facebook_20260824A
checkpoint("final")

print("\nto resume in a fresh session: run the config cell and the checkpoint cell,\n"
      "leave RESTORE_FROM_CHECKPOINT = True, then run down the notebook. Page\n"
      "discovery skips, submitted batches are not resubmitted, and polling\n"
      "reattaches to the jobs Meta is still holding.")
